In [ ]:
# -*- coding: utf-8 -*-
"""
ÖĞRENCİ BAĞIMLILIK ANALİZİ - AYRI GRAFİKLERLE TAM ÇÖZÜM
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report, 
                           confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import LabelEncoder

# 1. VERİ YÜKLEME VE HAZIRLIK
def load_and_prepare():
    train = pd.read_csv("student_addiction_dataset_train.csv")
    test = pd.read_csv("student_addiction_dataset_test.csv")
    df = pd.concat([train, test])
    
    # Kategorik değişkenleri kodla
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    
    # Hedef değişken (son sütun varsayılıyor)
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    
    return train_test_split(X, y, test_size=0.2, random_state=42)

# 2. MODEL EĞİTİMİ
def train_gb_model(X_train, y_train):
    model = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.1,
        max_depth=4,
        min_samples_split=5,
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

# 3. GRAFİK FONKSİYONLARI (AYRI AYRI)
def plot_confusion_matrix(y_true, y_pred):
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred), 
                annot=True, fmt='d', cmap='Blues',
                annot_kws={"size": 14})
    plt.title('Confusion Matrix')
    plt.show()

def plot_roc_curve(y_true, y_proba):
    plt.figure(figsize=(8, 6))
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'AUC = {auc(fpr, tpr):.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Eğrisi')
    plt.legend(loc="lower right")
    plt.show()

def plot_feature_importance(model, feature_names):
    plt.figure(figsize=(10, 8))
    importances = model.feature_importances_
    indices = np.argsort(importances)[-10:]
    plt.barh(range(len(indices)), importances[indices], color='b', align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.title('Özellik Önem Sıralaması (Top 10)')
    plt.show()

def plot_prediction_distribution(y_proba):
    plt.figure(figsize=(8, 6))
    sns.histplot(y_proba, bins=20, kde=True, color='green')
    plt.title('Tahmin Olasılıkları Dağılımı')
    plt.show()

def plot_correlation_matrix(X):
    plt.figure(figsize=(12, 10))
    corr = X.corr()
    sns.heatmap(corr, cmap='coolwarm', center=0, 
                annot=True, fmt='.1f', annot_kws={"size": 10})
    plt.title('Korelasyon Matrisi')
    plt.show()

def plot_top_features_boxplot(X, y, top_features):
    plt.figure(figsize=(10, 6))
    for i, feature in enumerate(top_features, 1):
        plt.subplot(1, len(top_features), i)
        sns.boxplot(x=y, y=X[feature])
        plt.title(f'"{feature}" Dağılımı')
    plt.tight_layout()
    plt.show()

# 4. ANA İŞLEM
def main():
    print("⏳ Veri yükleniyor ve hazırlanıyor...")
    X_train, X_test, y_train, y_test = load_and_prepare()
    
    print("🚀 Yüksek doğruluklu model eğitiliyor...")
    model = train_gb_model(X_train, y_train)
    
    # Tahminler
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Performans
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n✅ SONUÇ: Doğruluk Oranı = {accuracy:.2%}")
    print("\n📊 Detaylı Performans Raporu:")
    print(classification_report(y_test, y_pred))
    
    # Grafikler
    print("\n📈 Grafikler oluşturuluyor...")
    
    # 1. Confusion Matrix
    plot_confusion_matrix(y_test, y_pred)
    
    # 2. ROC Curve
    plot_roc_curve(y_test, y_proba)
    
    # 3. Feature Importance
    plot_feature_importance(model, X_test.columns)
    
    # 4. Prediction Distribution
    plot_prediction_distribution(y_proba)
    
    # 5. Correlation Matrix
    plot_correlation_matrix(X_test)
    
    # 6. Top Features Boxplot
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-3:]
    top_features = [X_test.columns[i] for i in top_indices]
    plot_top_features_boxplot(X_test, y_test, top_features)

if __name__ == "__main__":
    main()

⏳ Veri yükleniyor ve hazırlanıyor...
❌ Kritik hata: Multi-dimensional indexing (e.g. `obj[:, None]`) is no longer supported. Convert to a numpy array before indexing instead.
Lütfen veri dosyalarını kontrol edin
